# Практика 3. Нейросетевые представления видео



## 0. Среда и воспроизводимость

Нужна GPU. После `%pip install`
перезапустите runtime, если импорт сообщает о несовместимых версиях.


In [ ]:
!pip install -q "torch>=2.2" "torchvision>=0.17" "transformers>=4.39"                 "huggingface_hub>=0.23" "av>=12" "scikit-learn>=1.4"                 "pandas>=2.0" "matplotlib>=3.8" "tqdm>=4.66"                 "pytorchvideo>=0.1.5" "fvcore>=0.1.5"

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import random
import tarfile
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Iterable, Literal

import av
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet50_Weights, resnet50
from torchvision.models.video import R2Plus1D_18_Weights, r2plus1d_18
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FAST_MODE = True
RUN_FINETUNE = False
RUN_PROTOCOL_MATRIX = True
RUN_VIDEOMAE_PROBE = True
RUN_X3D_PROBE = False

CWD = Path.cwd().resolve()
PRACTICE_DIR = CWD
DATA_DIR = CWD / "data"
CACHE_DIR = CWD / "features"
DATA_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

print("device:", DEVICE)
print("practice:", PRACTICE_DIR)

## 1. Данные и временное сэмплирование

Архив содержит 300 train,
30 validation и 75 test-видео


In [ ]:
def safe_extract(archive_path: Path, destination: Path) -> None:
    '''Extract a tar archive without allowing path traversal.'''
    destination = destination.resolve()
    with tarfile.open(archive_path) as archive:
        for member in archive.getmembers():
            target = (destination / member.name).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f"Unsafe archive member: {member.name}")
        archive.extractall(destination)


def prepare_ucf_subset(destination: Path = DATA_DIR) -> Path:
    '''Download and return the root of the 10-class UCF101 subset.'''
    candidates = [destination / "UCF101_subset", destination / "ucf101_subset"]
    for candidate in candidates:
        if (candidate / "train").exists():
            return candidate
    archive = Path(hf_hub_download(
        repo_id="sayakpaul/ucf101-subset",
        filename="UCF101_subset.tar.gz",
        repo_type="dataset",
    ))
    safe_extract(archive, destination)
    roots = [p.parent for p in destination.rglob("train") if p.is_dir()]
    if not roots:
        raise FileNotFoundError("После распаковки не найдена папка train")
    return roots[0]


UCF_ROOT = prepare_ucf_subset()
for split in ("train", "val", "test"):
    split_dir = UCF_ROOT / split

    empty_dirs = sorted(
        (path for path in split_dir.rglob("*") if path.is_dir()),
        key=lambda path: len(path.parts),
        reverse=True,
    )
    for directory in empty_dirs:
        if not any(directory.iterdir()):
            directory.rmdir()

counts = {
    split: sum(path.is_file() for path in (UCF_ROOT / split).rglob("*"))
    for split in ("train", "val", "test")
}

print(UCF_ROOT, counts)
assert counts == {"train": 300, "val": 30, "test": 75}, counts

In [ ]:
def probe_video(path: Path) -> dict[str, object]:
    '''Read container metadata without decoding all frames.'''
    with av.open(str(path)) as container:
        stream = container.streams.video[0]
        fps = float(stream.average_rate) if stream.average_rate else np.nan
        n_frames = int(stream.frames or 0)
        duration = n_frames / fps if fps and n_frames else np.nan
        return {
            "file": path.name,
            "class": path.parent.name,
            "fps": fps,
            "frames": n_frames,
            "duration_s": duration,
            "width": stream.width,
            "height": stream.height,
        }


sample_paths = sorted((UCF_ROOT / "train").glob("*/*"))[:30]
metadata = pd.DataFrame(probe_video(p) for p in tqdm(sample_paths))
display(metadata.head())
display(metadata.describe(include="all").round(2))


### 1.1. Состав датасета
Сначала посмотрим на данные до загрузки моделей.

In [ ]:
split_names = ("train", "val", "test")
class_names = sorted(p.name for p in (UCF_ROOT / "train").iterdir() if p.is_dir())

class_count_rows = []
metadata_paths = []
for split in split_names:
    for class_name in class_names:
        paths = sorted(
            p for p in (UCF_ROOT / split / class_name).glob("*") if p.is_file()
        )
        class_count_rows.append({
            "split": split,
            "class": class_name,
            "videos": len(paths),
        })
        metadata_paths.extend(paths[:3])

class_counts = pd.DataFrame(class_count_rows)
metadata_visual = pd.DataFrame(probe_video(path) for path in tqdm(metadata_paths))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

class_counts.pivot(index="class", columns="split", values="videos").plot(
    kind="bar", ax=axes[0, 0], color=["#287271", "#E9C46A", "#E76F51"]
)
axes[0, 0].set_title("Число видео по классам и split")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("Видео")
axes[0, 0].tick_params(axis="x", rotation=45)

axes[0, 1].hist(
    metadata_visual["duration_s"].dropna(), bins=15,
    color="#2A9D8F", edgecolor="white",
)
axes[0, 1].set_title("Распределение длительности")
axes[0, 1].set_xlabel("Секунды")
axes[0, 1].set_ylabel("Видео")

fps_counts = metadata_visual["fps"].round(2).value_counts().sort_index()
axes[1, 0].bar(fps_counts.index.astype(str), fps_counts.values, color="#457B9D")
axes[1, 0].set_title("Частота кадров")
axes[1, 0].set_xlabel("FPS")
axes[1, 0].set_ylabel("Видео")

resolution_counts = (
    metadata_visual.assign(
        resolution=lambda frame: (
            frame["width"].astype(str) + "×" + frame["height"].astype(str)
        )
    )["resolution"].value_counts()
)
axes[1, 1].barh(resolution_counts.index, resolution_counts.values, color="#F4A261")
axes[1, 1].set_title("Разрешения кадров")
axes[1, 1].set_xlabel("Видео")

fig.suptitle("UCF101-subset: проверка состава до обучения", fontsize=15)
fig.tight_layout()
display(class_counts.pivot(index="class", columns="split", values="videos"))


### Задание 1. Индексы кадров

Реализуйте три стратегии для `total` кадров:

- `uniform`: равномерно покрывает весь ролик;
- `dense`: берёт соседние кадры вокруг центра;
- `strided`: берёт кадры с заданным шагом и дополняет последним кадром.


In [ ]:
Sampling = Literal["uniform", "dense", "strided"]

def sample_indices(
    total: int,
    num_frames: int,
    strategy: Sampling,
    stride: int = 1,
    center: float = 0.5,
) -> np.ndarray:
    '''Return exactly ``num_frames`` valid frame indices.'''
    if total < 1 or num_frames < 1:
        raise ValueError("total and num_frames must be positive")
    if strategy == "uniform":
        idx = np.linspace(0, total - 1, num_frames)
    else:
        step = 1 if strategy == "dense" else stride
        span = step * (num_frames - 1)
        anchor = int(round(center * (total - 1)))
        start = max(0, min(anchor - span // 2, total - 1 - span))
        idx = start + step * np.arange(num_frames)
    return np.clip(np.rint(idx), 0, total - 1).astype(np.int64)

assert sample_indices(10, 4, "strided", stride=4).tolist() == [0, 4, 8, 9]

In [ ]:
def decode_video(path: Path) -> np.ndarray:
    '''Decode a video to uint8 RGB frames with shape [N, H, W, C].'''
    frames: list[np.ndarray] = []
    with av.open(str(path)) as container:
        for frame in container.decode(video=0):
            frames.append(frame.to_rgb().to_ndarray())
    if not frames:
        raise ValueError(f"No frames decoded from {path}")
    return np.stack(frames)


def resize_and_crop(
    frames: torch.Tensor,
    size: int = 224,
    crop_position: Literal["left", "center", "right"] = "center",
) -> torch.Tensor:
    '''Resize the short side, then apply one consistent square crop.'''
    _, _, height, width = frames.shape
    scale = size / min(height, width)
    new_h, new_w = math.ceil(height * scale), math.ceil(width * scale)
    frames = F.interpolate(frames, size=(new_h, new_w), mode="bilinear", align_corners=False)
    top = (new_h - size) // 2
    if crop_position == "left":
        left = 0
    elif crop_position == "right":
        left = new_w - size
    else:
        left = (new_w - size) // 2
    return frames[:, :, top:top + size, left:left + size]


def load_clip(
    path: Path,
    num_frames: int,
    strategy: Sampling = "uniform",
    stride: int = 1,
    center: float = 0.5,
    crop_position: Literal["left", "center", "right"] = "center",
) -> torch.Tensor:
    '''Load a normalized-to-[0,1] clip as [T, C, 224, 224].'''
    decoded = decode_video(path)
    idx = sample_indices(len(decoded), num_frames, strategy, stride, center)
    clip = torch.from_numpy(decoded[idx]).permute(0, 3, 1, 2).float() / 255.0
    return resize_and_crop(clip, 224, crop_position)


def augment_clip(clip: torch.Tensor) -> torch.Tensor:
    '''Apply one spatial crop and flip consistently to every frame.'''
    size = clip.shape[-1]
    crop_size = random.randint(int(0.8 * size), size)
    top = random.randint(0, size - crop_size)
    left = random.randint(0, size - crop_size)
    clip = clip[:, :, top:top + crop_size, left:left + crop_size]
    clip = F.interpolate(clip, size=(size, size), mode="bilinear", align_corners=False)
    if random.random() < 0.5:
        clip = clip.flip(-1)
    return clip


example_path = sorted((UCF_ROOT / "train").glob("*/*"))[0]
decoded = decode_video(example_path)
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
strategies = [("uniform", 1), ("dense", 1), ("strided", 4)]
for row, (strategy, stride) in enumerate(strategies):
    idx = sample_indices(len(decoded), 8, strategy, stride)
    for ax, frame, frame_idx in zip(axes[row], decoded[idx], idx):
        ax.imshow(frame)
        ax.set_title(str(frame_idx))
        ax.axis("off")
    axes[row, 0].set_ylabel(strategy)
plt.tight_layout()


### 1.2. Классы, временной охват и аугментации
Верхняя галерея показывает центральный кадр каждого класса. Затем одна
шкала времени сравнивает фактические индексы uniform, dense и
strided. Последний график проверяет, что пространственная аугментация
применяется согласованно ко всему клипу.

In [ ]:
class_examples = [
    sorted((UCF_ROOT / "train" / class_name).glob("*"))[0]
    for class_name in class_names
]

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for ax, path in zip(axes.flat, class_examples):
    frames = decode_video(path)
    ax.imshow(frames[len(frames) // 2])
    ax.set_title(path.parent.name)
    ax.axis("off")
fig.suptitle("По одному центральному кадру каждого класса", fontsize=15)
fig.tight_layout()

timeline_video = decode_video(class_examples[-2])
timeline_modes = [("uniform", 1), ("dense", 1), ("strided", 4)]
fig, ax = plt.subplots(figsize=(14, 3.5))
for level, (strategy, stride) in enumerate(timeline_modes):
    indices = sample_indices(len(timeline_video), 16, strategy, stride)
    ax.scatter(indices, np.full_like(indices, level), s=45, label=strategy)
    for order, frame_index in enumerate(indices):
        ax.text(frame_index, level + 0.08, str(order), fontsize=7, ha="center")
ax.set_yticks(range(len(timeline_modes)), [mode[0] for mode in timeline_modes])
ax.set_xlabel("Индекс исходного кадра; подпись над точкой — позиция во входном клипе")
ax.set_title(f"Временной охват одного видео: {class_examples[-2].parent.name}")
ax.grid(axis="x", alpha=0.25)
plt.show()

original_clip = load_clip(class_examples[-2], 16, "uniform")
random_state = random.getstate()
augmented_clips = [augment_clip(original_clip.clone()) for _ in range(3)]
random.setstate(random_state)
compared_clips = [original_clip] + augmented_clips
titles = ["без аугментации", "crop + flip #1", "crop + flip #2", "crop + flip #3"]

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for column, (clip, title) in enumerate(zip(compared_clips, titles)):
    for row, frame_index in enumerate((3, 12)):
        axes[row, column].imshow(clip[frame_index].permute(1, 2, 0).clamp(0, 1))
        axes[row, column].set_title(title if row == 0 else f"кадр {frame_index}")
        axes[row, column].axis("off")
fig.suptitle("Одна пространственная трансформация применяется ко всем кадрам клипа")
fig.tight_layout()


Какая стратегия видит порядок локального движения?
Какая гарантированно замечает событие в любой части ролика?



## 2. Паспорт моделей: форма, параметры, цена

В ноутбуке все батчи хранятся как `[B, T, C, H, W]`. Адаптеры сами
переставляют оси для torchvision (`[B, C, T, H, W]`).

- **ResNet-50:** K. He et al., [Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385), CVPR 2016.
- **R(2+1)D:** D. Tran et al., [A Closer Look at Spatiotemporal Convolutions for Action Recognition](https://arxiv.org/abs/1711.11248), CVPR 2018.
- **X3D:** C. Feichtenhofer, [X3D: Expanding Architectures for Efficient Video Recognition](https://arxiv.org/abs/2004.04730), CVPR 2020.
- **VideoMAE:** Z. Tong et al., [VideoMAE: Masked Autoencoders are Data-Efficient Learners for Self-Supervised Video Pre-Training](https://arxiv.org/abs/2203.12602), NeurIPS 2022.


In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3, 1, 1)
R2_MEAN = torch.tensor([0.43216, 0.394666, 0.37645]).view(1, 1, 3, 1, 1)
R2_STD = torch.tensor([0.22803, 0.22145, 0.216989]).view(1, 1, 3, 1, 1)
X3D_MEAN = torch.tensor([0.45, 0.45, 0.45]).view(1, 1, 3, 1, 1)
X3D_STD = torch.tensor([0.225, 0.225, 0.225]).view(1, 1, 3, 1, 1)


class ResNetMeanEncoder(nn.Module):
    '''ImageNet ResNet-50 applied per frame, followed by temporal mean.'''

    def __init__(self) -> None:
        super().__init__()
        self.backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.backbone.fc = nn.Identity()
        self.output_dim = 2048

    def forward(self, clips: torch.Tensor) -> torch.Tensor:
        b, t, c, h, w = clips.shape
        x = (clips - IMAGENET_MEAN.to(clips)) / IMAGENET_STD.to(clips)
        features = self.backbone(x.reshape(b * t, c, h, w))
        return features.view(b, t, -1).mean(dim=1)


class R2Plus1DEncoder(nn.Module):
    '''Kinetics-400 R(2+1)D-18 without its classification head.'''

    def __init__(self) -> None:
        super().__init__()
        self.backbone = r2plus1d_18(weights=R2Plus1D_18_Weights.KINETICS400_V1)
        self.output_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

    def forward(self, clips: torch.Tensor) -> torch.Tensor:
        b, t, c, h, w = clips.shape
        x = F.interpolate(
            clips.reshape(b * t, c, h, w), size=(112, 112),
            mode="bilinear", align_corners=False,
        ).view(b, t, c, 112, 112)
        x = (x - R2_MEAN.to(x)) / R2_STD.to(x)
        return self.backbone(x.permute(0, 2, 1, 3, 4))


class VideoMAEEncoder(nn.Module):
    '''VideoMAE base encoder with mean pooling over output tokens.'''

    def __init__(self, checkpoint: str = "MCG-NJU/videomae-base") -> None:
        super().__init__()
        from transformers import VideoMAEModel
        self.backbone = VideoMAEModel.from_pretrained(checkpoint)
        self.output_dim = self.backbone.config.hidden_size

    def forward(self, clips: torch.Tensor) -> torch.Tensor:
        x = (clips - IMAGENET_MEAN.to(clips)) / IMAGENET_STD.to(clips)
        return self.backbone(pixel_values=x).last_hidden_state.mean(dim=1)


class X3DEncoder(nn.Module):
    '''Kinetics-400 X3D-S features captured before the final projection.'''

    def __init__(self) -> None:
        super().__init__()
        self.backbone = torch.hub.load(
            "facebookresearch/pytorchvideo", "x3d_s",
            pretrained=True, trust_repo=True,
        )

    def forward(self, clips: torch.Tensor) -> torch.Tensor:
        b, t, c, h, w = clips.shape
        x = F.interpolate(
            clips.reshape(b * t, c, h, w), size=(160, 160),
            mode="bilinear", align_corners=False,
        ).view(b, t, c, 160, 160)
        x = (x - X3D_MEAN.to(x)) / X3D_STD.to(x)
        captured: list[torch.Tensor] = []
        handle = self.backbone.blocks[-1].pool.register_forward_hook(
            lambda _module, _inputs, output: captured.append(output)
        )
        try:
            self.backbone(x.permute(0, 2, 1, 3, 4))
        finally:
            handle.remove()
        if not captured:
            raise RuntimeError("X3D feature hook did not fire")
        return captured[0].flatten(1)


def build_encoder(name: str) -> nn.Module:
    '''Build one feature extractor; keep only one large model on GPU.'''
    builders = {
        "resnet50_mean": ResNetMeanEncoder,
        "r2plus1d_18": R2Plus1DEncoder,
        "x3d_s": X3DEncoder,
        "videomae_b": VideoMAEEncoder,
    }
    if name not in builders:
        raise KeyError(f"Unknown encoder: {name}")
    return builders[name]().eval().to(DEVICE)



In [ ]:
@torch.inference_mode()
def model_card(model: nn.Module, clip_shape: tuple[int, ...], repeats: int = 10) -> dict[str, float]:
    '''Measure parameters and median forward latency after warm-up.'''
    sample = torch.rand(clip_shape, device=DEVICE)
    for _ in range(2):
        model(sample)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        model(sample)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        timings.append((time.perf_counter() - start) * 1000)
    return {
        "params_m": sum(p.numel() for p in model.parameters()) / 1e6,
        "latency_ms": float(np.median(timings)),
    }


encoder = build_encoder("resnet50_mean")
print(model_card(encoder, (1, 8, 3, 224, 224)))
del encoder
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


## 3. Zero-shot: что Kinetics уже знает про UCF101

Предобученная голова выдаёт 400 меток Kinetics. Мы не переобучаем её,
а смотрим top-5 на нескольких UCF101-роликах.


In [ ]:
KINETICS_HINTS = {
    "ApplyEyeMakeup": ["applying makeup"],
    "ApplyLipstick": ["putting on lipstick", "applying makeup"],
    "Archery": ["archery"],
    "BabyCrawling": ["crawling baby"],
    "BalanceBeam": ["gymnastics tumbling"],
    "BandMarching": ["marching"],
    "BaseballPitch": ["playing baseball"],
    "Basketball": ["playing basketball"],
    "BasketballDunk": ["dunking basketball"],
    "BenchPress": ["bench pressing"],
}


@torch.inference_mode()
def videomae_top5(model: nn.Module, path: Path) -> list[tuple[str, float]]:
    '''Return the five most likely Kinetics labels for one UCF clip.'''
    clip = load_clip(path, 16, "uniform").unsqueeze(0).to(DEVICE)
    pixels = (clip - IMAGENET_MEAN.to(clip)) / IMAGENET_STD.to(clip)
    probs = model(pixel_values=pixels).logits.softmax(dim=-1)[0]
    values, indices = probs.topk(5)
    return [(model.config.id2label[int(i)], float(v)) for v, i in zip(values, indices)]


from transformers import VideoMAEForVideoClassification
zero_shot_model = VideoMAEForVideoClassification.from_pretrained(
    "MCG-NJU/videomae-base-finetuned-kinetics"
).eval().to(DEVICE)
demo_paths = [sorted(class_dir.glob("*"))[0]
              for class_dir in sorted((UCF_ROOT / "test").iterdir()) if class_dir.is_dir()]
zero_shot_rows = []
for path in tqdm(demo_paths):
    predictions = videomae_top5(zero_shot_model, path)
    expected = set(KINETICS_HINTS.get(path.parent.name, []))
    predicted_labels = {label for label, _ in predictions}
    zero_shot_rows.append({
        "ucf_class": path.parent.name,
        "top5": ", ".join(label for label, _ in predictions),
        "direct_or_synonym_hit": bool(expected & predicted_labels),
    })
zero_shot_table = pd.DataFrame(zero_shot_rows)
examples = pd.concat([
    zero_shot_table[zero_shot_table.direct_or_synonym_hit].head(3),
    zero_shot_table[~zero_shot_table.direct_or_synonym_hit].head(3),
])
display(examples)
del zero_shot_model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


Какие успехи объясняет объект или сцена без движения?
Какие ошибки связаны с несовпадающей таксономией?


## 4. Замороженные признаки и linear probe


In [ ]:
class VideoFolderDataset(Dataset):
    '''Deterministic subset of a class-folder video dataset.'''

    def __init__(
        self,
        root: Path,
        split: str,
        num_frames: int,
        strategy: Sampling,
        stride: int = 1,
        max_per_class: int | None = None,
        augment: bool = False,
    ) -> None:
        self.num_frames = num_frames
        self.strategy = strategy
        self.stride = stride
        self.augment = augment
        classes = sorted(p.name for p in (root / "train").iterdir() if p.is_dir())
        self.class_to_idx = {name: idx for idx, name in enumerate(classes)}
        self.samples: list[tuple[Path, int]] = []
        for class_name in classes:
            paths = sorted((root / split / class_name).glob("*"))
            if max_per_class is not None:
                paths = paths[:max_per_class]
            self.samples.extend((p, self.class_to_idx[class_name]) for p in paths)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int, str]:
        path, label = self.samples[index]
        clip = load_clip(path, self.num_frames, self.strategy, self.stride)
        if self.augment:
            clip = augment_clip(clip)
        return clip, label, str(path)


def make_loader(
    split: str,
    num_frames: int,
    strategy: Sampling = "uniform",
    stride: int = 1,
    batch_size: int = 4,
    augment: bool = False,
) -> DataLoader:
    limits = {"train": 10, "val": 3, "test": 5} if FAST_MODE else {}
    dataset = VideoFolderDataset(
        UCF_ROOT, split, num_frames, strategy, stride,
        max_per_class=limits.get(split),
        augment=augment,
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)


@torch.inference_mode()
def extract_features(model: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray, list[str]]:
    '''Extract one feature vector for every clip.'''
    features, labels, paths = [], [], []
    model.eval()
    for clips, y, batch_paths in tqdm(loader, leave=False):
        z = model(clips.to(DEVICE, non_blocking=True))
        features.append(z.cpu().numpy())
        labels.append(y.numpy())
        paths.extend(batch_paths)
    return np.concatenate(features), np.concatenate(labels), paths


def cache_key(name: str, split: str, frames: int, strategy: str, stride: int) -> Path:
    mode = "fast" if FAST_MODE else "full"
    return CACHE_DIR / f"{name}_{split}_t{frames}_{strategy}_s{stride}_{mode}.npz"


def cached_features(
    name: str,
    split: str,
    frames: int,
    strategy: Sampling = "uniform",
    stride: int = 1,
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    '''Load features from disk or compute them once.'''
    target = cache_key(name, split, frames, strategy, stride)
    if target.exists():
        data = np.load(target, allow_pickle=False)
        return data["X"], data["y"], data["paths"].tolist()
    model = build_encoder(name)
    result = extract_features(model, make_loader(split, frames, strategy, stride))
    np.savez_compressed(target, X=result[0], y=result[1], paths=np.asarray(result[2]))
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return result


def fit_probe(X_train: np.ndarray, y_train: np.ndarray) -> object:
    '''Fit a standardized multinomial logistic-regression head.'''
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, C=1.0, random_state=SEED),
    ).fit(X_train, y_train)


`VideoFolderDataset` возвращает клипы в едином формате
`[B, T, C, H, W]`. Ниже показаны несколько кадров каждого элемента
случайного батча, метки и диапазоны значений. Перед конкретной моделью
адаптер меняет layout и применяет нормализацию:

- ResNet-50: `[B, T, C, H, W] → [B·T, C, H, W]`;
- R(2+1)D и X3D: `[B, T, C, H, W] → [B, C, T, H, W]`;
- VideoMAE: сохраняет `[B, T, C, H, W]`.

In [ ]:
batch_dataset = VideoFolderDataset(
    UCF_ROOT, "train", num_frames=16, strategy="uniform", max_per_class=3
)
batch_generator = torch.Generator().manual_seed(SEED)
batch_loader = DataLoader(
    batch_dataset, batch_size=4, shuffle=True,
    generator=batch_generator, num_workers=0,
)
clips_batch, labels_batch, paths_batch = next(iter(batch_loader))
idx_to_class = {index: name for name, index in batch_dataset.class_to_idx.items()}

print("shape [B,T,C,H,W]:", tuple(clips_batch.shape))
print("dtype:", clips_batch.dtype)
print("диапазон:", float(clips_batch.min()), float(clips_batch.max()))
display(pd.DataFrame({
    "label_id": labels_batch.tolist(),
    "class": [idx_to_class[int(label)] for label in labels_batch],
    "file": [Path(path).name for path in paths_batch],
}))

shown_frames = np.linspace(0, clips_batch.shape[1] - 1, 5).astype(int)
fig, axes = plt.subplots(len(clips_batch), len(shown_frames), figsize=(15, 10))
for row in range(len(clips_batch)):
    for column, frame_index in enumerate(shown_frames):
        axes[row, column].imshow(
            clips_batch[row, frame_index].permute(1, 2, 0).clamp(0, 1)
        )
        axes[row, column].set_title(
            f"{idx_to_class[int(labels_batch[row])]}\nt={frame_index}"
        )
        axes[row, column].axis("off")
fig.suptitle("Содержимое батча до model-specific нормализации", fontsize=15)
fig.tight_layout()

normalized_versions = {
    "raw [0,1]": clips_batch,
    "ResNet / VideoMAE": (
        clips_batch - IMAGENET_MEAN
    ) / IMAGENET_STD,
    "R(2+1)D": (clips_batch - R2_MEAN) / R2_STD,
}
batch_stats = pd.DataFrame([
    {
        "representation": name,
        "mean": float(tensor.mean()),
        "std": float(tensor.std()),
        "min": float(tensor.min()),
        "max": float(tensor.max()),
    }
    for name, tensor in normalized_versions.items()
])
display(batch_stats.round(3))


### Задание 2. Один кадр против нескольких
Для ResNet-50 извлеките признаки при `T=1` и `T=8`, обучите две
логистические головы и заполните таблицу. Рассчитайте
`appearance_ratio = accuracy(T=1) / accuracy(T=8)`.


In [ ]:
probe_rows: list[dict[str, object]] = []
fitted_probes: dict[str, object] = {}

for frames in (1, 8):
    name = "resnet50_mean"
    X_train, y_train, _ = cached_features(name, "train", frames)
    X_test, y_test, _ = cached_features(name, "test", frames)
    probe = fit_probe(X_train, y_train)
    accuracy = accuracy_score(y_test, probe.predict(X_test))
    key = f"resnet_t{frames}"
    fitted_probes[key] = probe
    probe_rows.append({
        "model": "ResNet-50 mean",
        "frames": frames,
        "feature_dim": X_train.shape[1],
        "accuracy": accuracy,
        "GFLOPs/view": 4.09 * frames,
    })

probe_table = pd.DataFrame(probe_rows)
acc_1 = float(probe_table.loc[probe_table.frames == 1, "accuracy"].iloc[0])
acc_8 = float(probe_table.loc[probe_table.frames == 8, "accuracy"].iloc[0])
print(f"appearance_ratio = {acc_1 / max(acc_8, 1e-12):.3f}")
display(probe_table)



### R(2+1)D и VideoMAE

Теперь меняется сам энкодер. Для честного сравнения сохраняем один
split, число примеров и linear-probe алгоритм. У моделей остаются их
штатные число кадров, нормализация и crop.


In [ ]:
video_models = ["r2plus1d_18"]
if RUN_X3D_PROBE:
    video_models.append("x3d_s")
if RUN_VIDEOMAE_PROBE:
    video_models.append("videomae_b")
model_cost = {"r2plus1d_18": 40.52, "x3d_s": 1.96, "videomae_b": 180.0}
model_title = {
    "r2plus1d_18": "R(2+1)D-18", "x3d_s": "X3D-S",
    "videomae_b": "VideoMAE-B SSL",
}

for name in video_models:
    frames = 13 if name == "x3d_s" else 16
    X_train, y_train, _ = cached_features(name, "train", frames)
    X_test, y_test, _ = cached_features(name, "test", frames)
    probe = fit_probe(X_train, y_train)
    fitted_probes[name] = probe
    probe_rows.append({
        "model": model_title[name],
        "frames": frames,
        "feature_dim": X_train.shape[1],
        "accuracy": accuracy_score(y_test, probe.predict(X_test)),
        "GFLOPs/view": model_cost[name],
    })

probe_table = pd.DataFrame(probe_rows).sort_values("accuracy", ascending=False)
display(probe_table.style.format({"accuracy": "{:.3f}", "GFLOPs/view": "{:.2f}"}))


Self-supervised VideoMAE оптимизировали на восстановление
скрытых пикселей, поэтому его признаки не обязаны быть линейно разделимыми
без дообучения.


Одного среднего accuracy недостаточно. Confusion matrix показывает пары
смешиваемых действий, accuracy по классам обнаруживает слабые категории,
а PCA даёт качественную картину разделимости замороженных признаков.
PCA служит только визуализацией: linear probe обучается в исходном пространстве.

In [ ]:
from sklearn.decomposition import PCA

diagnostic_specs = [
    ("ResNet T=1", "resnet50_mean", 1, "resnet_t1"),
    ("ResNet T=8", "resnet50_mean", 8, "resnet_t8"),
    ("R(2+1)D-18", "r2plus1d_18", 16, "r2plus1d_18"),
]
if "videomae_b" in fitted_probes:
    diagnostic_specs.append(("VideoMAE-B SSL", "videomae_b", 16, "videomae_b"))
if "x3d_s" in fitted_probes:
    diagnostic_specs.append(("X3D-S", "x3d_s", 13, "x3d_s"))

diagnostics = {}
for title, encoder_name, frames, probe_key in diagnostic_specs:
    X_test_diag, y_test_diag, paths_diag = cached_features(
        encoder_name, "test", frames, "uniform"
    )
    probe_diag = fitted_probes[probe_key]
    predictions_diag = probe_diag.predict(X_test_diag)
    diagnostics[title] = {
        "X": X_test_diag,
        "y": y_test_diag,
        "pred": predictions_diag,
        "paths": paths_diag,
        "probe": probe_diag,
    }

n_models = len(diagnostics)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4.8), squeeze=False)
for ax, (title, values) in zip(axes[0], diagnostics.items()):
    matrix = confusion_matrix(
        values["y"], values["pred"], labels=np.arange(len(class_names))
    )
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_title(title)
    ax.set_xticks(range(len(class_names)), class_names, rotation=90, fontsize=8)
    ax.set_yticks(range(len(class_names)), class_names, fontsize=8)
    ax.set_xlabel("Предсказание")
    ax.set_ylabel("Истинный класс")
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            ax.text(column, row, matrix[row, column], ha="center", va="center", fontsize=7)
    fig.colorbar(image, ax=ax, fraction=0.046)
fig.suptitle("Confusion matrices замороженных представлений", fontsize=15)
fig.tight_layout()

per_class_rows = []
for title, values in diagnostics.items():
    for class_index, class_name in enumerate(class_names):
        mask = values["y"] == class_index
        per_class_rows.append({
            "model": title,
            "class": class_name,
            "accuracy": float(np.mean(values["pred"][mask] == class_index)),
        })
per_class_table = pd.DataFrame(per_class_rows).pivot(
    index="class", columns="model", values="accuracy"
)
ax = per_class_table.plot(kind="bar", figsize=(15, 5), ylim=(0, 1.05))
ax.set_title("Accuracy по классам после linear probe")
ax.set_ylabel("Accuracy")
ax.set_xlabel("")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
display(per_class_table.round(2))


In [ ]:
fig, axes = plt.subplots(1, len(diagnostics), figsize=(5 * len(diagnostics), 4.5), squeeze=False)
color_map = plt.get_cmap("tab10")
for ax, (title, values) in zip(axes[0], diagnostics.items()):
    scaler = values["probe"].named_steps["standardscaler"]
    standardized = scaler.transform(values["X"])
    embedding = PCA(n_components=2, random_state=SEED).fit_transform(standardized)
    for class_index, class_name in enumerate(class_names):
        mask = values["y"] == class_index
        ax.scatter(
            embedding[mask, 0], embedding[mask, 1], s=28,
            alpha=0.75, color=color_map(class_index), label=class_name,
        )
    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.grid(alpha=0.2)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.0, 0.5))
fig.suptitle("PCA test-признаков: геометрия классов после предобучения", fontsize=15)
fig.tight_layout(rect=(0, 0, 0.88, 1))


## 5. Полный файнтюнинг VideoMAE-B


In [ ]:
def normalize_videomae(clips: torch.Tensor) -> torch.Tensor:
    return (clips - IMAGENET_MEAN.to(clips)) / IMAGENET_STD.to(clips)


def fine_tune_videomae(
    epochs: int = 1 if FAST_MODE else 4,
    batch_size: int = 2 if FAST_MODE else 4,
    accumulation: int = 8,
) -> tuple[nn.Module, pd.DataFrame]:
    '''Fine-tune VideoMAE with mixed precision and gradient accumulation.'''
    from transformers import VideoMAEForVideoClassification

    classes = sorted(p.name for p in (UCF_ROOT / "train").iterdir() if p.is_dir())
    id2label = {i: name for i, name in enumerate(classes)}
    label2id = {name: i for i, name in id2label.items()}
    model = VideoMAEForVideoClassification.from_pretrained(
        "MCG-NJU/videomae-base",
        num_labels=len(classes),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    ).to(DEVICE)
    train_loader = make_loader(
        "train", 16, "uniform", batch_size=batch_size, augment=True
    )
    val_loader = make_loader("val", 16, "uniform", batch_size=batch_size)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.05)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    history = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        for step, (clips, labels, _) in enumerate(tqdm(train_loader, leave=False), 1):
            clips, labels = clips.to(DEVICE), labels.to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
                loss = model(pixel_values=normalize_videomae(clips), labels=labels).loss
                scaled_loss = loss / accumulation
            scaler.scale(scaled_loss).backward()
            if step % accumulation == 0 or step == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            running_loss += float(loss.detach())

        model.eval()
        pred, true = [], []
        with torch.inference_mode():
            for clips, labels, _ in val_loader:
                logits = model(pixel_values=normalize_videomae(clips.to(DEVICE))).logits
                pred.extend(logits.argmax(1).cpu().tolist())
                true.extend(labels.tolist())
        history.append({
            "epoch": epoch + 1,
            "train_loss": running_loss / max(len(train_loader), 1),
            "val_accuracy": accuracy_score(true, pred),
        })
    return model, pd.DataFrame(history)


finetuned_videomae = None

finetuned_videomae, finetune_history = fine_tune_videomae()
display(finetune_history)


Ниже показаны динамика loss и validation accuracy, итоговая confusion
matrix на test, сравнение с замороженным VideoMAE и примеры ошибок.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(finetune_history["epoch"], finetune_history["train_loss"], marker="o")
axes[0].set_title("Train loss")
axes[0].set_xlabel("Эпоха")
axes[0].set_ylabel("Cross-entropy")
axes[0].grid(alpha=0.25)
axes[1].plot(
    finetune_history["epoch"], finetune_history["val_accuracy"],
    marker="o", color="#E76F51",
)
axes[1].set_title("Validation accuracy")
axes[1].set_xlabel("Эпоха")
axes[1].set_ylim(0, 1.05)
axes[1].grid(alpha=0.25)
fig.suptitle("Динамика полного файнтюнинга VideoMAE")
fig.tight_layout()


@torch.inference_mode()
def evaluate_finetuned_videomae(
    model: nn.Module, loader: DataLoader
) -> tuple[np.ndarray, np.ndarray, np.ndarray, list[str]]:
    model.eval()
    probabilities, labels, paths = [], [], []
    for clips, batch_labels, batch_paths in tqdm(loader, leave=False):
        logits = model(
            pixel_values=normalize_videomae(clips.to(DEVICE))
        ).logits
        probabilities.append(logits.softmax(dim=-1).cpu().numpy())
        labels.append(batch_labels.numpy())
        paths.extend(batch_paths)
    probs = np.concatenate(probabilities)
    true = np.concatenate(labels)
    return probs, true, probs.argmax(axis=1), paths


finetune_test_loader = make_loader("test", 16, "uniform", batch_size=2)
ft_probs, ft_true, ft_pred, ft_paths = evaluate_finetuned_videomae(
    finetuned_videomae, finetune_test_loader
)
ft_accuracy = accuracy_score(ft_true, ft_pred)
ft_matrix = confusion_matrix(ft_true, ft_pred, labels=np.arange(len(class_names)))

fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(ft_matrix, cmap="Purples")
ax.set_xticks(range(len(class_names)), class_names, rotation=90)
ax.set_yticks(range(len(class_names)), class_names)
ax.set_xlabel("Предсказание")
ax.set_ylabel("Истинный класс")
ax.set_title(f"VideoMAE после fine-tuning: test accuracy = {ft_accuracy:.3f}")
for row in range(ft_matrix.shape[0]):
    for column in range(ft_matrix.shape[1]):
        ax.text(column, row, ft_matrix[row, column], ha="center", va="center", fontsize=8)
fig.colorbar(image, ax=ax, fraction=0.046)
fig.tight_layout()

comparison_rows = [{"режим": "VideoMAE full fine-tuning", "accuracy": ft_accuracy}]
ssl_rows = probe_table[probe_table["model"] == "VideoMAE-B SSL"]
if not ssl_rows.empty:
    comparison_rows.insert(0, {
        "режим": "VideoMAE frozen + linear probe",
        "accuracy": float(ssl_rows.iloc[0]["accuracy"]),
    })
finetune_comparison = pd.DataFrame(comparison_rows)
ax = finetune_comparison.plot(
    kind="bar", x="режим", y="accuracy", legend=False,
    figsize=(8, 4), ylim=(0, 1.05), color=["#457B9D", "#E76F51"],
)
ax.set_title("Замороженные признаки и полный fine-tuning")
ax.set_xlabel("")
ax.set_ylabel("Test accuracy")
ax.tick_params(axis="x", rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f")
plt.tight_layout()
display(finetune_comparison)


In [ ]:
error_indices = np.flatnonzero(ft_pred != ft_true)
if len(error_indices) == 0:
    # Если ошибок нет, показываем наименее уверенные правильные ответы.
    error_indices = np.argsort(ft_probs.max(axis=1))[:6]
    gallery_title = "Наименее уверенные правильные ответы"
else:
    # Сначала показываем самые уверенные ошибки: они наиболее информативны.
    error_indices = error_indices[np.argsort(-ft_probs[error_indices].max(axis=1))][:6]
    gallery_title = "Наиболее уверенные ошибки после fine-tuning"

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax in axes.flat:
    ax.axis("off")
for ax, index in zip(axes.flat, error_indices):
    frames = decode_video(Path(ft_paths[index]))
    ax.imshow(frames[len(frames) // 2])
    confidence = ft_probs[index, ft_pred[index]]
    ax.set_title(
        f"true: {class_names[ft_true[index]]}\n"
        f"pred: {class_names[ft_pred[index]]} ({confidence:.2f})"
    )
    ax.axis("off")
fig.suptitle(gallery_title, fontsize=15)
fig.tight_layout()


## 6. Сдвиг протокола сэмплирования


### Задание 3. Матрица train × test

Для R(2+1)D обучите linear probe на `uniform` и `dense` признаках,
затем проверьте каждую голову на обоих вариантах test. Интерпретируйте
строки и столбцы отдельно.


In [ ]:
protocol_rows = []
for train_strategy in ("uniform", "dense"):
    X_train, y_train, _ = cached_features("r2plus1d_18", "train", 16, train_strategy)
    probe = fit_probe(X_train, y_train)
    for test_strategy in ("uniform", "dense"):
        X_test, y_test, _ = cached_features("r2plus1d_18", "test", 16, test_strategy)
        protocol_rows.append({
            "train": train_strategy,
            "test": test_strategy,
            "accuracy": accuracy_score(y_test, probe.predict(X_test)),
        })

protocol_table = pd.DataFrame(protocol_rows)
if not protocol_table.empty:
    display(protocol_table.pivot(index="train", columns="test", values="accuracy").round(3))
display(protocol_table.pivot(index="train", columns="test", values="accuracy"))


In [ ]:
protocol_matrix = protocol_table.pivot(
    index="train", columns="test", values="accuracy"
)
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(protocol_matrix.values, cmap="YlGn", vmin=0, vmax=1)
ax.set_xticks(range(len(protocol_matrix.columns)), protocol_matrix.columns)
ax.set_yticks(range(len(protocol_matrix.index)), protocol_matrix.index)
ax.set_xlabel("Sampling на test")
ax.set_ylabel("Sampling на train")
ax.set_title("Согласование train/test-протоколов")
for row in range(protocol_matrix.shape[0]):
    for column in range(protocol_matrix.shape[1]):
        ax.text(
            column, row, f"{protocol_matrix.iloc[row, column]:.3f}",
            ha="center", va="center", fontsize=12,
        )
fig.colorbar(image, ax=ax, label="Accuracy")
fig.tight_layout()


Повторите test для `strided` с шагами
2 и 4. Проверьте долю повторённых индексов в коротких роликах: высокая
доля означает, что номинально длинный временной охват не реализовался.


## 7. Test-time views и полная вычислительная цена


In [ ]:
def temporal_centers(n_clips: int) -> np.ndarray:
    '''Return deterministic center positions in [0, 1].'''
    return np.array([0.5]) if n_clips == 1 else np.linspace(0.1, 0.9, n_clips)


@torch.inference_mode()
def predict_views(
    encoder: nn.Module,
    probe: object,
    path: Path,
    n_clips: int,
    n_crops: int,
) -> tuple[np.ndarray, float]:
    '''Average probe logits over temporal clips and spatial crops.'''
    if n_crops not in (1, 3):
        raise ValueError("n_crops must be 1 or 3")
    crops = ["center"] if n_crops == 1 else ["left", "center", "right"]
    views = []
    for center in temporal_centers(n_clips):
        for crop in crops:
            views.append(load_clip(path, 16, "dense", center=float(center), crop_position=crop))
    batch = torch.stack(views).to(DEVICE)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    features = encoder(batch).cpu().numpy()
    logits = probe.decision_function(features)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elapsed_ms = (time.perf_counter() - start) * 1000
    return np.asarray(logits).mean(axis=0), elapsed_ms


def evaluate_views(
    probe: object,
    settings: Iterable[tuple[int, int]],
    max_videos: int | None = None,
) -> pd.DataFrame:
    '''Evaluate multi-view R(2+1)D inference on test videos.'''
    class_count = len([p for p in (UCF_ROOT / "test").iterdir() if p.is_dir()])
    max_per_class = None if max_videos is None else max(1, max_videos // class_count)
    dataset = VideoFolderDataset(
        UCF_ROOT, "test", 16, "dense", max_per_class=max_per_class
    )
    samples = dataset.samples
    encoder = build_encoder("r2plus1d_18")
    rows = []
    for n_clips, n_crops in settings:
        pred, true, elapsed = [], [], []
        for path, label in tqdm(samples, leave=False):
            logits, ms = predict_views(encoder, probe, path, n_clips, n_crops)
            pred.append(int(np.argmax(logits)))
            true.append(label)
            elapsed.append(ms)
        views = n_clips * n_crops
        rows.append({
            "model": "R(2+1)D-18",
            "protocol": f"{n_clips}x{n_crops}",
            "views": views,
            "accuracy": accuracy_score(true, pred),
            "GFLOPs_x_views": 40.52 * views,
            "ms_per_video": float(np.mean(elapsed)),
        })
    del encoder
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return pd.DataFrame(rows)


### Задание 4. Ещё одна точка на графике
Возьмите R(2+1)D probe, обученный на `uniform`, и измерьте 1×1 и 3×1.
Затем добавьте 5×3.


In [ ]:
X_train_r2, y_train_r2, _ = cached_features("r2plus1d_18", "train", 16, "uniform")
r2_probe = fit_probe(X_train_r2, y_train_r2)
view_settings = [(1, 1), (3, 1)] + ([] if FAST_MODE else [(5, 3)])
views_table = evaluate_views(r2_probe, view_settings, max_videos=20 if FAST_MODE else None)
display(views_table.style.format({
    "accuracy": "{:.3f}", "GFLOPs_x_views": "{:.1f}", "ms_per_video": "{:.1f}",
}))

In [ ]:
fig, ax_accuracy = plt.subplots(figsize=(8, 4.5))
ordered_views = views_table.sort_values("views")
ax_accuracy.plot(
    ordered_views["views"], ordered_views["accuracy"],
    marker="o", linewidth=2, color="#2A9D8F", label="accuracy",
)
ax_accuracy.set_xlabel("Число views")
ax_accuracy.set_ylabel("Accuracy", color="#2A9D8F")
ax_accuracy.set_ylim(0, 1.05)
ax_accuracy.tick_params(axis="y", labelcolor="#2A9D8F")
ax_accuracy.grid(alpha=0.2)

ax_latency = ax_accuracy.twinx()
ax_latency.plot(
    ordered_views["views"], ordered_views["ms_per_video"],
    marker="s", linestyle="--", color="#E76F51", label="latency",
)
ax_latency.set_ylabel("мс на видео", color="#E76F51")
ax_latency.tick_params(axis="y", labelcolor="#E76F51")
ax_accuracy.set_title("Качество и задержка при увеличении числа views")
fig.tight_layout()


In [ ]:
def plot_pareto(rows: pd.DataFrame) -> None:
    '''Plot accuracy against total inference compute.'''
    fig, ax = plt.subplots(figsize=(9, 5))
    for _, row in rows.iterrows():
        ax.scatter(row["GFLOPs_x_views"], row["accuracy"], s=70)
        ax.annotate(
            f'{row["model"]} {row["protocol"]}',
            (row["GFLOPs_x_views"], row["accuracy"]),
            xytext=(5, 5), textcoords="offset points",
        )
    ax.set_xscale("log")
    ax.set_xlabel("GFLOPs на view × число views (лог-шкала)")
    ax.set_ylabel("Accuracy")
    ax.set_title("Качество и полная стоимость инференса")
    ax.grid(True, which="both", alpha=0.25)
    plt.show()


result_rows = []
if "probe_table" in globals() and not probe_table.empty:
    for _, row in probe_table.iterrows():
        result_rows.append({
            "model": row["model"], "protocol": "1x1", "views": 1,
            "accuracy": row["accuracy"], "GFLOPs_x_views": row["GFLOPs/view"],
            "ms_per_video": np.nan,
        })
if "views_table" in globals():
    result_rows.extend(views_table.to_dict("records"))

results = pd.DataFrame(result_rows)
if not results.empty:
    display(results)
    plot_pareto(results)
    results.to_csv(PRACTICE_DIR / "results.csv", index=False)
